# Stage 05 — Train Qwen2.5 LoRA

| | |
|---|---|
| **Intention** | LoRA fine-tune `Qwen/Qwen2.5-1.5B-Instruct` for text → gloss. |
| **Input** | `data/mbart/{train,dev}.jsonl` |
| **Output** | `artifacts/ckpts/qwen/best/` (PEFT adapter) |
| **Runtime** | ~1–2 h on GPU |


In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from ttg.config import DATA_DIR, CHECKPOINTS_DIR, ARTIFACTS, PROJECT_ROOT
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("CHECKPOINTS  =", CHECKPOINTS_DIR)


PROJECT_ROOT = /home/khurshida/Projects/uzsl-text-to-gloss
DATA_DIR     = /home/khurshida/Projects/uzsl-text-to-gloss/data
CHECKPOINTS  = /home/khurshida/Projects/uzsl-text-to-gloss/artifacts/ckpts


In [ ]:
SMOKE_TEST = False
BF16 = True
DIRECTION = "text2gloss"  # "text2gloss" or "gloss2text"
EPOCHS = 1 if SMOKE_TEST else 10
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4
MAX_LENGTH = 256
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
SEED = 42
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = CHECKPOINTS_DIR / ("qwen_g2t" if DIRECTION == "gloss2text" else "qwen")


In [3]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer
from ttg.config import USER_PROMPT, USER_PROMPT_GLOSS_TO_TEXT
from ttg.data import load_split, swap_direction, to_chat_messages

set_seed(SEED)
train = load_split(DATA_DIR / "mbart" / "train.jsonl")
dev = load_split(DATA_DIR / "mbart" / "dev.jsonl")
if DIRECTION == "gloss2text":
    train, dev = swap_direction(train), swap_direction(dev)
prompt = USER_PROMPT_GLOSS_TO_TEXT if DIRECTION == "gloss2text" else USER_PROMPT

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Register the fingerspelling marker as a real token so it survives training
# and generation intact instead of being fragmented into subwords.
tokenizer.add_tokens(["[dct]"], special_tokens=False)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16 if BF16 and torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model.config.use_cache = False
# Some base checkpoints pad their embedding table beyond the tokenizer's real
# vocab size (e.g. for hardware alignment); only grow, never shrink it, or
# rows for genuine (if unused-by-us) token ids would be silently discarded.
if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
    model.resize_token_embeddings(len(tokenizer))
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    # LoRA freezes the base embedding/head; without this the new token's
    # embedding row stays at random init and is never actually learned.
    modules_to_save=["embed_tokens", "lm_head"],
    bias="none",
))
model.print_trainable_parameters()

train_ds = Dataset.from_list(to_chat_messages(train, prompt))
dev_ds = Dataset.from_list(to_chat_messages(dev, prompt))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, num_train_epochs=EPOCHS,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, logging_steps=5,
        save_total_limit=1, save_only_model=True,
        seed=SEED, bf16=BF16 and torch.cuda.is_available(),
        max_length=MAX_LENGTH, assistant_only_loss=True, report_to=[],
    ),
    train_dataset=train_ds, eval_dataset=dev_ds, processing_class=tokenizer,
)
trainer.train()
best_dir = OUTPUT_DIR / "best"
best_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
meta = {"model": MODEL, "direction": DIRECTION, "num_train": len(train.texts), "num_dev": len(dev.texts),
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "lr": LR, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "user_prompt": prompt}
(OUTPUT_DIR / "train_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved →", best_dir)


/home/khurshida/miniforge3/envs/uzsl-ttg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 9615.59it/s]

/home/khurshida/miniforge3/envs/uzsl-ttg/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


...


Truncating eval dataset:   0%|          | 0/130 [00:00<?, ? examples/s]


Truncating eval dataset: 100%|██████████| 130/130 [00:00<00:00, 14429.44 examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.749722,1.726415,1.662165,147187.000000,0.650550
2,1.130174,1.604969,1.193500,294374.000000,0.684753
3,0.668844,1.716957,0.865105,441561.000000,0.686649
4,0.283087,2.004777,0.562874,588748.000000,0.685151
5,0.136987,2.120000,0.492423,735935.000000,0.684858
6,0.066896,2.228674,0.432623,883122.000000,0.689806
7,0.035217,2.308075,0.392671,1030309.000000,0.689039
8,0.008602,2.363631,0.374904,1177496.000000,0.687739
9,0.004459,2.444374,0.341581,1324683.000000,0.695349
10,0.001933,2.457199,0.341638,1471870.000000,0.694676


Saved → /home/khurshida/Projects/uzsl-text-to-gloss/artifacts/ckpts/qwen_g2t/best
